# ML-07 — Baseline Action Score and Top-10 Review

**Lane locked:** Refresh / Content Opportunity Scoring. This notebook builds one deliberately simple, public-safe review rule on the starter FlyRank snapshot. The rule is a baseline for Week 5, not a claim about Google's algorithm or the causal effect of an edit.

> Working with an AI assistant? Read `skills/README.md`, then load `building-baselines` + `flyrank/flyrank-data`.

## 1. Check two signals before writing the rule

**Rule idea in plain words.** Review pages that already have enough search exposure, rank within the first 20 positions, and capture unusually few clicks. Rank those candidates by the size of the observable CTR gap, the amount of exposure at stake, and how close the page is to the top of the results.

The two checks below use `is_declining_proxy = (trend_direction == "down")` only as an outcome for audit and evaluation. Neither `trend_direction` nor `trend_pct` enters the score. Rates are percentage points: `ctr = 0.50` means 0.50%.

### Signal 1 — CTR versus position (FlyRank flag-linked)

The real CTR-fix logic assumes that, among pages with measurable volume and similar visible positions, lower CTR can identify a review opportunity. I restrict the check to at least 300 impressions and positions 1–20, then bucket CTR into four observed groups.

**Verdict: CONFIRMED.** The lowest-CTR bucket has the highest observed decline share, and the share falls across the four buckets. This supports a review flag, not a causal claim that changing metadata will reverse the decline.

### Signal 2 — search volume

Volume is the evidence/impact signal behind quick-win logic. I check fixed volume bands among pages in positions 1–20.

**Verdict: MIXED.** The relationship is not monotonic: the 300–2,999 group has the highest observed decline share, while the 30,000+ group has the lowest. Volume therefore belongs in this rule as an **impact priority and minimum evidence floor**, not as proof that a page is more likely to decline.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

# Find the repository root in Colab or a local clone.
repo_root = Path.cwd()
while not (repo_root / "data" / "raw" / "content_refresh_anonymized.csv").exists():
    if repo_root == repo_root.parent:
        raise FileNotFoundError("Could not locate the starter dataset.")
    repo_root = repo_root.parent

df = pd.read_csv(repo_root / "data" / "raw" / "content_refresh_anonymized.csv")
df["is_declining_proxy"] = df["trend_direction"].eq("down").astype("int8")

# Signal 1: CTR quartiles within an evidence-controlled visible slice.
ctr_slice = df[
    df["impressions_90d"].ge(300)
    & df["avg_position"].gt(0)
    & df["avg_position"].le(20)
].copy()
ctr_slice["ctr_bucket"] = pd.qcut(ctr_slice["ctr"], q=4, duplicates="drop")
ctr_bucket_table = (
    ctr_slice.groupby("ctr_bucket", observed=True)
    .agg(
        n=("content_id", "size"),
        median_ctr_pct=("ctr", "median"),
        median_position=("avg_position", "median"),
        observed_decline_share=("is_declining_proxy", "mean"),
    )
    .reset_index()
)
ctr_bucket_table["observed_decline_share"] = ctr_bucket_table["observed_decline_share"].round(3)
print("Signal 1 bucket table — CTR vs position (n printed)")
display(ctr_bucket_table)
print("Verdict: CONFIRMED")

# Signal 2: fixed exposure bands inside the same visible position range.
volume_slice = df[df["avg_position"].gt(0) & df["avg_position"].le(20)].copy()
volume_slice["volume_bucket"] = pd.cut(
    volume_slice["impressions_90d"],
    bins=[0, 300, 3_000, 30_000, np.inf],
    labels=["1-299", "300-2,999", "3,000-29,999", "30,000+"],
    right=False,
    include_lowest=True,
)
volume_bucket_table = (
    volume_slice.groupby("volume_bucket", observed=True)
    .agg(
        n=("content_id", "size"),
        median_impressions=("impressions_90d", "median"),
        median_ctr_pct=("ctr", "median"),
        observed_decline_share=("is_declining_proxy", "mean"),
    )
    .reset_index()
)
volume_bucket_table["observed_decline_share"] = volume_bucket_table["observed_decline_share"].round(3)
print("\nSignal 2 bucket table — volume (n printed)")
display(volume_bucket_table)
print("Verdict: MIXED")

Signal 1 bucket table — CTR vs position (n printed)


,ctr_bucket,n,median_ctr_pct,median_position,observed_decline_share
0,"(-0.001, 0.09]",3455,0.00,9.8,0.724
1,"(0.09, 0.21]",3312,0.15,8.6,0.610
2,"(0.21, 0.41]",3231,0.29,7.6,0.582
3,"(0.41, 5.43]",3207,0.64,7.3,0.503


Verdict: CONFIRMED

Signal 2 bucket table — volume (n printed)


,volume_bucket,n,median_impressions,median_ctr_pct,observed_decline_share
0,1-299,7051,31.0,0.00,0.529
1,"300-2,999",6899,1089.0,0.17,0.671
2,"3,000-29,999",5463,7175.0,0.25,0.555
3,"30,000+",843,49350.0,0.26,0.421


Verdict: MIXED


## 2. Encode one transparent rule and write the ranked queue

**Eligibility:** at least 300 impressions, measured average position from 1 through 20, and CTR below 0.50 percentage points.

**Score:** `log(1 + impressions) × (0.50 − CTR) × ((21 − position) / 20)`. These are hand-written weights and thresholds, not fitted coefficients. The score grows when the observable CTR gap, exposure at stake, or position opportunity grows.

- **One reason code:** `visible_low_ctr_for_position`
- **One action label:** `review_title_meta_and_intent`

The queue is written to `work/outputs/baseline_action_score.csv`. It is intentionally ignored by git and regenerated on every run. A small JSON receipt is committed.

In [2]:
eligible = (
    df["impressions_90d"].ge(300)
    & df["avg_position"].gt(0)
    & df["avg_position"].le(20)
    & df["ctr"].lt(0.50)
)

queue = df.loc[eligible, [
    "content_id", "impressions_90d", "clicks_90d", "ctr", "avg_position",
    "days_since_last_update", "is_declining_proxy",
]].copy()
queue["baseline_score"] = (
    np.log1p(queue["impressions_90d"])
    * (0.50 - queue["ctr"])
    * ((21 - queue["avg_position"]) / 20)
)
queue["reason_code"] = "visible_low_ctr_for_position"
queue["action_label"] = "review_title_meta_and_intent"
queue = queue.sort_values(
    ["baseline_score", "impressions_90d", "content_id"],
    ascending=[False, False, True],
    kind="stable",
).reset_index(drop=True)
queue.insert(0, "rank", np.arange(1, len(queue) + 1))

output_dir = repo_root / "work" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)
queue_path = output_dir / "baseline_action_score.csv"
queue.to_csv(queue_path, index=False)

def precision_at_k(frame, k):
    return float(frame.head(k)["is_declining_proxy"].mean())

base_rate = float(df["is_declining_proxy"].mean())
metrics = {
    "lane": "Refresh / Content Opportunity Scoring",
    "data": "starter snapshot: one row per pseudonymized content item",
    "rule": "visible positions 1-20, impressions >= 300, CTR < 0.50%; rank by exposure-weighted CTR gap",
    "eligible_rows": int(len(queue)),
    "base_rate": base_rate,
    "precision_at_10": precision_at_k(queue, 10),
    "precision_at_50": precision_at_k(queue, 50),
    "reason_code": "visible_low_ctr_for_position",
    "action_label": "review_title_meta_and_intent",
    "score_inputs": ["impressions_90d", "ctr", "avg_position"],
    "excluded_label_fields": ["trend_direction", "trend_pct"],
}
receipt_path = output_dir / "w04_baseline_metrics.json"
receipt_path.write_text(json.dumps(metrics, indent=2) + "\n")

print("Queue rows:", f"{len(queue):,}")
print("Base rate:", f"{base_rate:.1%}")
print("Precision@10:", f"{metrics['precision_at_10']:.1%}")
print("Precision@50:", f"{metrics['precision_at_50']:.1%}")
print("Wrote:", queue_path.relative_to(repo_root))
print("Wrote:", receipt_path.relative_to(repo_root))
display(queue.head(10))

Queue rows: 10,730
Base rate: 54.2%
Precision@10: 50.0%
Precision@50: 70.0%
Wrote: work/outputs/baseline_action_score.csv
Wrote: work/outputs/w04_baseline_metrics.json


,rank,content_id,impressions_90d,clicks_90d,ctr,avg_position,days_since_last_update,is_declining_proxy,baseline_score,reason_code,action_label
0,1,content_8451fc6f034d,272144,75,0.03,2.3,20,0,5.499317,visible_low_ctr_for_position,review_title_meta_and_intent
1,2,content_4a6607efcb46,128068,17,0.01,2.2,104,0,5.416805,visible_low_ctr_for_position,review_title_meta_and_intent
2,3,content_d225ec9f3d46,26470,14,0.05,0.7,20,1,4.651453,visible_low_ctr_for_position,review_title_meta_and_intent
3,4,content_e12868d1f396,149712,104,0.07,2.9,7,0,4.637296,visible_low_ctr_for_position,review_title_meta_and_intent
4,5,content_339b357d04c7,46879,7,0.01,3.7,15,0,4.558654,visible_low_ctr_for_position,review_title_meta_and_intent
5,6,content_0022a6b4290f,29747,22,0.07,1.2,20,1,4.384930,visible_low_ctr_for_position,review_title_meta_and_intent
6,7,content_954cc45bd437,15439,6,0.04,1.5,7,1,4.325655,visible_low_ctr_for_position,review_title_meta_and_intent
7,8,content_f4e210ee0c27,24784,16,0.06,1.6,20,1,4.318360,visible_low_ctr_for_position,review_title_meta_and_intent
8,9,content_cbdf5a78dcd0,14830,3,0.02,2.4,104,0,4.287438,visible_low_ctr_for_position,review_title_meta_and_intent
9,10,content_fea6a0d13b4a,79965,56,0.07,3.4,104,1,4.271893,visible_low_ctr_for_position,review_title_meta_and_intent


## 3. Top-10 skeptical review

Each line below records the same proposed action, why the item ranked highly, and a concrete condition that would make the recommendation wrong. These are review hypotheses based on aggregate, pseudonymized measurements—not diagnoses of page quality.

In [3]:
wrong_if = [
    "the query mix is branded or a SERP feature satisfies the intent without a click",
    "the position average combines unrelated queries and the low CTR disappears by query",
    "the sub-position-1 average is a reporting artifact or the result is not a standard organic listing",
    "the page already matches intent and the apparent gap comes from zero-click search behavior",
    "a small set of high-impression queries dominates the average and needs query-level review first",
    "the listing already changed during the 90-day aggregate and the current title is not the measured title",
    "the traffic is seasonal or navigational, making a metadata edit irrelevant",
    "the page serves a deliberate low-click informational answer and clicks are not the right success measure",
    "the average position masks unstable rankings across days or devices",
    "the CTR gap is caused by intent mismatch that requires a different page, not a metadata edit",
]

top10_review = queue.head(10).copy()
top10_review["review_line"] = [
    f"#{int(row['rank'])}: ACTION review title/meta and intent; WHY {int(row['impressions_90d']):,} impressions, "
    f"position {row['avg_position']:.1f}, CTR {row['ctr']:.2f}%; WRONG IF {wrong_if[i]}."
    for i, (_, row) in enumerate(top10_review.iterrows())
]

for line in top10_review["review_line"]:
    print(line)

display(top10_review[[
    "rank", "content_id", "action_label", "reason_code", "baseline_score", "review_line"
]])

#1: ACTION review title/meta and intent; WHY 272,144 impressions, position 2.3, CTR 0.03%; WRONG IF the query mix is branded or a SERP feature satisfies the intent without a click.
#2: ACTION review title/meta and intent; WHY 128,068 impressions, position 2.2, CTR 0.01%; WRONG IF the position average combines unrelated queries and the low CTR disappears by query.
#3: ACTION review title/meta and intent; WHY 26,470 impressions, position 0.7, CTR 0.05%; WRONG IF the sub-position-1 average is a reporting artifact or the result is not a standard organic listing.
#4: ACTION review title/meta and intent; WHY 149,712 impressions, position 2.9, CTR 0.07%; WRONG IF the page already matches intent and the apparent gap comes from zero-click search behavior.
#5: ACTION review title/meta and intent; WHY 46,879 impressions, position 3.7, CTR 0.01%; WRONG IF a small set of high-impression queries dominates the average and needs query-level review first.
#6: ACTION review title/meta and intent; WHY 29

,rank,content_id,action_label,reason_code,baseline_score,review_line
0,1,content_8451fc6f034d,review_title_meta_and_intent,visible_low_ctr_for_position,5.499317,#1: ACTION review title/meta and intent; WHY 2...
1,2,content_4a6607efcb46,review_title_meta_and_intent,visible_low_ctr_for_position,5.416805,#2: ACTION review title/meta and intent; WHY 1...
2,3,content_d225ec9f3d46,review_title_meta_and_intent,visible_low_ctr_for_position,4.651453,#3: ACTION review title/meta and intent; WHY 2...
3,4,content_e12868d1f396,review_title_meta_and_intent,visible_low_ctr_for_position,4.637296,#4: ACTION review title/meta and intent; WHY 1...
4,5,content_339b357d04c7,review_title_meta_and_intent,visible_low_ctr_for_position,4.558654,#5: ACTION review title/meta and intent; WHY 4...
5,6,content_0022a6b4290f,review_title_meta_and_intent,visible_low_ctr_for_position,4.384930,#6: ACTION review title/meta and intent; WHY 2...
6,7,content_954cc45bd437,review_title_meta_and_intent,visible_low_ctr_for_position,4.325655,#7: ACTION review title/meta and intent; WHY 1...
7,8,content_f4e210ee0c27,review_title_meta_and_intent,visible_low_ctr_for_position,4.318360,#8: ACTION review title/meta and intent; WHY 2...
8,9,content_cbdf5a78dcd0,review_title_meta_and_intent,visible_low_ctr_for_position,4.287438,#9: ACTION review title/meta and intent; WHY 1...
9,10,content_fea6a0d13b4a,review_title_meta_and_intent,visible_low_ctr_for_position,4.271893,#10: ACTION review title/meta and intent; WHY ...


## 4. Weak picks and leakage check

**Weak picks found.** Several top-ten pages sit around positions 1–3. Their very low CTR is visually striking, but that can be misleading: branded/navigational queries, featured snippets, knowledge panels, blended result types, or query-mix averaging can suppress clicks without implying that the title or content is weak. The rule has no query-level context and cannot see whether the measured listing changed during the 90-day window. Those are reasons to review—not reasons to auto-edit.

**Why the rule remains honestly beatable.** It uses one fixed threshold and one multiplicative score for every content type and client. It does not learn client norms, query intent, seasonality, or interactions. Week 5 must compare a learned model against this frozen queue on the same rows, target, and Precision@50.

**Leakage boundary.** The score uses only `impressions_90d`, `ctr`, and `avg_position`. It does not use `trend_direction`, `trend_pct`, `is_declining_proxy`, any product flag, IDs, or a future-window column. The proxy is attached only after ranking to audit signals and compute Precision@K. Because the starter file is one trailing-window snapshot rather than a true past→future panel, results are observational teaching evidence; the capstone's warehouse validation must use non-overlapping pre/post windows.

In [4]:
score_inputs = {"impressions_90d", "ctr", "avg_position"}
forbidden_inputs = {
    "trend_direction", "trend_pct", "is_declining_proxy",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "content_id", "client_id",
}

assert score_inputs.isdisjoint(forbidden_inputs)
assert queue["content_id"].is_unique
assert queue["baseline_score"].is_monotonic_decreasing
assert queue["reason_code"].nunique() == 1
assert queue["action_label"].nunique() == 1
assert len(top10_review) == 10 and top10_review["review_line"].str.contains("WRONG IF").all()
assert queue_path.exists() and receipt_path.exists()

print("Score inputs:", sorted(score_inputs))
print("Forbidden/label-derived inputs used in score:", sorted(score_inputs & forbidden_inputs))
print("One reason code:", queue["reason_code"].unique().tolist())
print("One action label:", queue["action_label"].unique().tolist())
print("Top-10 skeptical lines:", len(top10_review))
print("Validation: PASS")

Score inputs: ['avg_position', 'ctr', 'impressions_90d']
Forbidden/label-derived inputs used in score: []
One reason code: ['visible_low_ctr_for_position']
One action label: ['review_title_meta_and_intent']
Top-10 skeptical lines: 10
Validation: PASS


## 5. Self-check

- [x] Lane is locked as Refresh / Content Opportunity Scoring.
- [x] Two signal checks have visible bucket tables and `n`; CTR-vs-position is flag-linked.
- [x] Each signal has a one-word verdict: CONFIRMED or MIXED.
- [x] One rule produces one score, one reason code, and one action label.
- [x] The notebook writes `work/outputs/baseline_action_score.csv` and a commit-safe JSON receipt.
- [x] Ten rows each state the action, why it ranked, and what would make it wrong.
- [x] No label-derived field, future-window field, product flag, or identifier enters the score.
- [x] Claims are observational and decision-support only; no client names, domains, URLs, or private queries appear.
- [x] The notebook runs top to bottom with visible outputs.